# PINN Vancomycin — Sensitivity Analysis (Kaggle, 2×T4)

Notebook za pokretanje glavnog eksperimenta na Kaggle GPU sesiji.

**Ključne razlike u odnosu na Colab verziju:**
- Nema Google Drive-a → rezultati idu u `/kaggle/working/` (automatski dostupni za download)
- 2×T4 GPU: `cuda:0` i `cuda:1` — 1-odeljni i 2-odeljni model rade **paralelno** (threading)
- Kaggle sesija traje do **12h** — dovoljno za oba modela paralelno

**Podešavanja pre pokretanja:**
1. `Settings → Accelerator → GPU T4 x2`
2. `Settings → Persistence → Variables and Files` (čuva `/kaggle/working/` između sesija)
3. U ćeliji 2: postavi `REPO_URL` na tvoj GitHub link

---
| Ćelija | Akcija | Trajanje |
|--------|--------|----------|
| 1 | GPU check + putanje | 5s |
| 2 | Kloniranje repoa + instalacija | 1–2 min |
| 3 | Generisanje podataka | 30s |
| 4 | Import modula | 5s |
| 5 | Test run (provjera) | ~5 min |
| 6 | **Paralelni eksperiment** (oba modela, oba GPU-a) | ~3–5h |
| 7 | Provjera napretka | 5s |
| 8 | Restore checkpoint | 10s |
| — | *(split sekcija — vidi dolje)* | — |

## Ćelija 1 — GPU check i putanje

In [ ]:
import torch
from pathlib import Path

n_gpu = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__}')
print(f'CUDA dostupna: {torch.cuda.is_available()}')
print(f'Broj GPU-a: {n_gpu}')
for i in range(n_gpu):
    print(f'  cuda:{i} → {torch.cuda.get_device_name(i)}')

if n_gpu < 2:
    print('\nUPOZORENJE: manje od 2 GPU-a. Provjeri Settings → Accelerator → GPU T4 x2.')

OUT_DIR  = Path('/kaggle/working/pinn_results')
REPO_DIR = Path('/kaggle/working/pinn-vancomycin')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'\nIzlazni folder: {OUT_DIR}')
print(f'Repo folder:    {REPO_DIR}')

## Ćelija 2 — Kloniranje repoa i instalacija

⚠️ **Izmijeni `REPO_URL`** na tvoj GitHub link.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'   # <-- IZMIJENI OVO
REPO_DIR = Path('/kaggle/working/pinn-vancomycin')

if not REPO_DIR.exists():
    !git clone $REPO_URL {REPO_DIR}
else:
    print('Repo vec postoji, povlacim promjene...')
    !git -C {REPO_DIR} pull

!git -C {REPO_DIR} log --oneline -3
!pip install scipy seaborn -q
print('\nInstalacija gotova.')

## Ćelija 3 — Generisanje sintetičkih podataka

In [ ]:
!python {REPO_DIR}/src/data_processing.py

## Ćelija 4 — Import modula i konfiguracija

In [ ]:
import sys
import importlib.util
import torch
from pathlib import Path

REPO_DIR = Path('/kaggle/working/pinn-vancomycin')
OUT_DIR  = Path('/kaggle/working/pinn_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_DIR / 'src'))

def _load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

sens = _load(REPO_DIR / 'experiments' / '03_sensitivity_analysis.py', 'sensitivity')

n_gpu   = torch.cuda.device_count()
DEVICE0 = 'cuda:0' if n_gpu >= 1 else 'cpu'
DEVICE1 = 'cuda:1' if n_gpu >= 2 else DEVICE0

print(f'1-odeljni model → {DEVICE0}')
print(f'2-odeljni model → {DEVICE1}')
print(f'Checkpoint folder: {OUT_DIR}')
print('Moduli učitani.')

## Ćelija 5 — Test run

Pokreće 8 kombinacija sa minimalnim brojem epoha (~5 min). Preporučeno pri prvom pokretanju.

In [ ]:
import threading, shutil

TEST_DIR = OUT_DIR / 'test'
TEST_DIR.mkdir(parents=True, exist_ok=True)
errors = []

def _test_1(): 
    try: sens.run_sensitivity_1comp(test_mode=True, out_dir=TEST_DIR, device=DEVICE0)
    except Exception as e: errors.append(f'1comp: {e}')

def _test_2():
    try: sens.run_sensitivity_2comp(test_mode=True, out_dir=TEST_DIR, device=DEVICE1)
    except Exception as e: errors.append(f'2comp: {e}')

t1, t2 = threading.Thread(target=_test_1), threading.Thread(target=_test_2)
t1.start(); t2.start()
t1.join();  t2.join()

if errors:
    for e in errors: print(f'GRESKA: {e}')
else:
    shutil.rmtree(TEST_DIR, ignore_errors=True)
    print('Test OK. Spreman za pravi eksperiment.')

## Ćelija 6 — Paralelni eksperiment (oba modela, oba GPU-a)

**1200 runova ukupno.** Procijenjeno trajanje (2×T4): **~3–5h** paralelno.

In [ ]:
import threading

errors = []

def _run_1comp():
    try:
        print(f'[1-comp] Start na {DEVICE0}')
        sens.run_sensitivity_1comp(test_mode=False, out_dir=OUT_DIR, device=DEVICE0)
        print('[1-comp] ZAVRSENO')
    except Exception as e:
        errors.append(f'1comp: {e}'); print(f'[1-comp] GRESKA: {e}')

def _run_2comp():
    try:
        print(f'[2-comp] Start na {DEVICE1}')
        sens.run_sensitivity_2comp(test_mode=False, out_dir=OUT_DIR, device=DEVICE1)
        print('[2-comp] ZAVRSENO')
    except Exception as e:
        errors.append(f'2comp: {e}'); print(f'[2-comp] GRESKA: {e}')

t1 = threading.Thread(target=_run_1comp)
t2 = threading.Thread(target=_run_2comp)
t1.start(); t2.start()
t1.join();  t2.join()

if errors:
    for e in errors: print(f'GRESKA: {e}')
else:
    print('Oba modela zavrsena. Rezultati su u Output tabu.')

## Ćelija 7 — Provjera napretka

Pokrenuti u bilo kom trenutku dok eksperiment radi.

In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path('/kaggle/working/pinn_results')
TOTAL   = 5 * 4 * 30

for name in ['sensitivity_1comp.csv', 'sensitivity_2comp.csv']:
    p = OUT_DIR / name
    if not p.exists():
        print(f'{name}: ne postoji jos'); continue
    df = pd.read_csv(p)
    ok = df[df['status'] == 'ok']
    print(f'{name}: {len(df)}/{TOTAL} ({len(df)/TOTAL*100:.1f}%)  uspjesnih={len(ok)}')
    if len(ok) > 0:
        print(f'  Median PINN err_k10: {ok["pinn_err_k10"].median():.2f}%')
        print(f'  Po N: {ok["N"].value_counts().sort_index().to_dict()}')
    print()

## Ćelija 8 — Restore checkpoint

Koristiti samo ako se sesija prekinula. Priloži prethodni CSV kao Kaggle Dataset  
(`Settings → Add data`), izmijeni `INPUT_DIR` ispod, pa pokreni ćelije 1–4 → ova ćelija → ćelija 6.

In [ ]:
import shutil
from pathlib import Path

OUT_DIR   = Path('/kaggle/working/pinn_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path('/kaggle/input/pinn-checkpoint')   # <-- izmijeni naziv dataseta

if not INPUT_DIR.exists():
    print(f'Dataset nije priložen: {INPUT_DIR}')
else:
    for csv in sorted(INPUT_DIR.glob('sensitivity_*.csv')):
        dst = OUT_DIR / csv.name
        shutil.copy(csv, dst)
        rows = sum(1 for _ in open(dst)) - 1
        print(f'  Restored: {csv.name}  ({rows} redova)')
    print('Restore zavrsen. Nastavi sa ćelijom 6.')

---
## SPLIT SEKCIJA — nastavak prekinutog 2-odeljnog eksperimenta

Koristiti kada imaš **djelimičan `sensitivity_2comp.csv`** (npr. N=3,5,8 gotovi)  
i hoćeš da **raspodijeliš preostale N vrednosti na oba GPU-a** da završiš brže.

**Redosljed:**
1. Preuzmi trenutni `sensitivity_2comp.csv` iz Output taba
2. Napravi novi Kaggle Dataset sa tim fajlom i priloži ga
3. Pokreni ćelije 1–4 (setup)
4. Pokreni **Split-S1** (restore djelimičnog CSV-a)
5. Pokreni **Split-S2** (paralelni run: N=10 na cuda:0, N=12 na cuda:1)
6. Pokreni **Split-S3** (merge svih CSV-ova u finalni fajl)

> Svaki GPU piše u **poseban CSV** (`split_N10/` i `split_N12/`) pa nema kolizije pri pisanju.

In [ ]:
# Split-S1 — Restore djelimičnog CSV-a u OUT_DIR
import shutil, pandas as pd
from pathlib import Path

OUT_DIR   = Path('/kaggle/working/pinn_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path('/kaggle/input/pinn-checkpoint')   # <-- izmijeni naziv dataseta

src = INPUT_DIR / 'sensitivity_2comp.csv'
if not src.exists():
    print(f'Fajl nije pronađen: {src}')
else:
    dst = OUT_DIR / 'sensitivity_2comp_partial.csv'
    shutil.copy(src, dst)
    df = pd.read_csv(dst)
    print(f'Učitan djelimični CSV: {len(df)} redova')
    print(f'N vrednosti koje su gotove: {sorted(df["N"].unique())}')
    print(f'N vrednosti koje nedostaju: {[n for n in [3,5,8,10,12] if n not in df["N"].unique()]}')

In [ ]:
# Split-S2 — Paralelni run: N=10 na cuda:0, N=12 na cuda:1
# Svaki GPU piše u poseban poddirektorij — nema kolizije pri pisanju
import threading
from pathlib import Path

OUT_DIR  = Path('/kaggle/working/pinn_results')
DIR_N10  = OUT_DIR / 'split_N10'
DIR_N12  = OUT_DIR / 'split_N12'
DIR_N10.mkdir(parents=True, exist_ok=True)
DIR_N12.mkdir(parents=True, exist_ok=True)

errors = []

def _run_N10():
    try:
        print(f'[N=10] Start na {DEVICE0}')
        sens.run_sensitivity_2comp(
            test_mode=False, out_dir=DIR_N10, device=DEVICE0, n_subset=[10]
        )
        print('[N=10] ZAVRSENO')
    except Exception as e:
        errors.append(f'N10: {e}'); print(f'[N=10] GRESKA: {e}')

def _run_N12():
    try:
        print(f'[N=12] Start na {DEVICE1}')
        sens.run_sensitivity_2comp(
            test_mode=False, out_dir=DIR_N12, device=DEVICE1, n_subset=[12]
        )
        print('[N=12] ZAVRSENO')
    except Exception as e:
        errors.append(f'N12: {e}'); print(f'[N=12] GRESKA: {e}')

t1 = threading.Thread(target=_run_N10)
t2 = threading.Thread(target=_run_N12)
t1.start(); t2.start()
t1.join();  t2.join()

if errors:
    for e in errors: print(f'GRESKA: {e}')
else:
    print('Oba N zavrsena.')

In [ ]:
# Split-S3 — Merge: djelimični + N10 + N12 → finalni sensitivity_2comp.csv
import pandas as pd
from pathlib import Path

OUT_DIR = Path('/kaggle/working/pinn_results')

parts = []
for p in [
    OUT_DIR / 'sensitivity_2comp_partial.csv',
    OUT_DIR / 'split_N10' / 'sensitivity_2comp.csv',
    OUT_DIR / 'split_N12' / 'sensitivity_2comp.csv',
]:
    if p.exists():
        df = pd.read_csv(p)
        print(f'  {p.name} (iz {p.parent.name}): {len(df)} redova, N={sorted(df["N"].unique())}')
        parts.append(df)
    else:
        print(f'  NEDOSTAJE: {p}')

if parts:
    final = pd.concat(parts, ignore_index=True)
    # Ukloni duplikate po (N, sigma, seed) ako ih ima
    before = len(final)
    final  = final.drop_duplicates(subset=['N', 'sigma', 'seed']).reset_index(drop=True)
    if before != len(final):
        print(f'  Uklonjeno {before - len(final)} duplikata')
    final = final.sort_values(['N', 'sigma', 'seed']).reset_index(drop=True)
    out = OUT_DIR / 'sensitivity_2comp.csv'
    final.to_csv(out, index=False)
    print(f'\nFinalni CSV: {len(final)}/600 redova → {out}')
    ok = final[final['status'] == 'ok']
    print(f'Uspjesnih runova: {len(ok)}/{len(final)}')
    print(f'Po N: {ok["N"].value_counts().sort_index().to_dict()}')